# PLP Data Analytics Assignment 2

## Rainfall Advisory Analysis

**Student:** Christine Wanjira

**Repository:** plp-da-a2-rainfall-advisory

### Project Objective

To analyse monthly rainfall data from five Kenyan counties, clean incorrect data, identify wettest and driest periods, and recommend suitable planting windows.

In [1]:
import numpy as np
import csv

In [2]:
counties = ["Nairobi", "Nakuru", "Kisumu", "Mombasa", "Machakos"]

clean_rows = [
    [64, 56, 92, 219, 176, 35, 18, 24, 31, 61, 165, 104],
    [35, 42, 70, 145, 120, 78, 96, 105, 72, 55, 60, 38],
    [58, 90, 150, 240, 180, 95, 70, 85, 90, 88, 130, 95],
    [30, 15, 55, 150, 240, 110, 90, 65, 60, 90, 100, 70]
]

machakos_raw = [
    "48", "35", "88", "N/A", "140", " 20 ",
    "9", "12", "", "45", "182", "96"
]

print("Counties:", counties)
print("Machakos raw data:", machakos_raw)

Counties: ['Nairobi', 'Nakuru', 'Kisumu', 'Mombasa', 'Machakos']
Machakos raw data: ['48', '35', '88', 'N/A', '140', ' 20 ', '9', '12', '', '45', '182', '96']


In [3]:
def clean_rainfall(value):
    try:
        return int(value.strip())
    except ValueError:
        return None

In [4]:
machakos_clean = [clean_rainfall(value) for value in machakos_raw]

print(machakos_clean)

[48, 35, 88, None, 140, 20, 9, 12, None, 45, 182, 96]


In [5]:
valid_values = [value for value in machakos_clean if value is not None]

imputed_value = round(np.mean(valid_values))

machakos_clean = [
    imputed_value if value is None else value
    for value in machakos_clean
]

print("Imputed value:", imputed_value)
print("Clean Machakos rainfall:", machakos_clean)

Imputed value: 68
Clean Machakos rainfall: [48, 35, 88, 68, 140, 20, 9, 12, 68, 45, 182, 96]


In [6]:
months = [
    "January", "February", "March", "April",
    "May", "June", "July", "August",
    "September", "October", "November", "December"
]

imputed_months = [
    months[i]
    for i, value in enumerate(machakos_raw)
    if clean_rainfall(value) is None
]

print("Machakos imputed months:", imputed_months)

Machakos imputed months: ['April', 'September']


In [7]:
rainfall_rows = clean_rows + [machakos_clean]

R = np.array(rainfall_rows)

print("Rainfall array:")
print(R)

print("Shape:", R.shape)
print("Data type:", R.dtype)

Rainfall array:
[[ 64  56  92 219 176  35  18  24  31  61 165 104]
 [ 35  42  70 145 120  78  96 105  72  55  60  38]
 [ 58  90 150 240 180  95  70  85  90  88 130  95]
 [ 30  15  55 150 240 110  90  65  60  90 100  70]
 [ 48  35  88  68 140  20   9  12  68  45 182  96]]
Shape: (5, 12)
Data type: int64


In [8]:
annual_totals = R.sum(axis=1)
monthly_averages = R.mean(axis=0)

print("Annual rainfall totals:")
for county, total in zip(counties, annual_totals):
    print(f"{county}: {total} mm")

print("\nAverage rainfall per month across all counties:")
for month, average in zip(months, monthly_averages):
    print(f"{month}: {average:.1f} mm")

Annual rainfall totals:
Nairobi: 1045 mm
Nakuru: 916 mm
Kisumu: 1371 mm
Mombasa: 1075 mm
Machakos: 811 mm

Average rainfall per month across all counties:
January: 47.0 mm
February: 47.6 mm
March: 91.0 mm
April: 164.4 mm
May: 171.2 mm
June: 67.6 mm
July: 56.6 mm
August: 58.2 mm
September: 64.2 mm
October: 67.8 mm
November: 127.4 mm
December: 80.6 mm


In [9]:
wettest_indices = np.argmax(R, axis=1)

wettest_months = [months[i] for i in wettest_indices]

print("Wettest month per county:")

for county, month in zip(counties, wettest_months):
    print(f"{county}: {month}")

Wettest month per county:
Nairobi: April
Nakuru: April
Kisumu: April
Mombasa: May
Machakos: November


In [10]:
dry_mask = R < 50

dry_month_counts = dry_mask.sum(axis=1)

print("Number of dry months per county:")

for county, count in zip(counties, dry_month_counts):
    print(f"{county}: {count} dry months")

Number of dry months per county:
Nairobi: 4 dry months
Nakuru: 3 dry months
Kisumu: 0 dry months
Mombasa: 2 dry months
Machakos: 6 dry months


In [11]:
two_month_totals = R[:, :-1] + R[:, 1:]

planting_start_indices = np.argmax(two_month_totals, axis=1)

planting_windows = [
    f"{months[i]}–{months[i+1]}"
    for i in planting_start_indices
]

planting_rainfall = two_month_totals[
    np.arange(len(counties)), planting_start_indices
]

print("Recommended planting windows:")

for county, window, total in zip(
    counties, planting_windows, planting_rainfall
):
    print(f"{county}: {window} ({total} mm)")

Recommended planting windows:
Nairobi: April–May (395 mm)
Nakuru: April–May (265 mm)
Kisumu: April–May (420 mm)
Mombasa: April–May (390 mm)
Machakos: November–December (278 mm)


In [12]:
with open("rainfall_summary.csv", "w", newline="") as file:
    fieldnames = [
        "county",
        "annual_total_mm",
        "wettest_month",
        "dry_months",
        "planting_window"
    ]

    writer = csv.DictWriter(file, fieldnames=fieldnames)

    writer.writeheader()

    for i, county in enumerate(counties):
        writer.writerow({
            "county": county,
            "annual_total_mm": int(annual_totals[i]),
            "wettest_month": wettest_months[i],
            "dry_months": int(dry_month_counts[i]),
            "planting_window": planting_windows[i]
        })

print("rainfall_summary.csv created successfully!")

rainfall_summary.csv created successfully!


In [13]:
with open("rainfall_summary.csv", "r") as file:
    reader = csv.DictReader(file)
    summary_rows = list(reader)

print("CSV columns:", reader.fieldnames)
print("Number of county records:", len(summary_rows))
print("\nCSV contents:")

for row in summary_rows:
    print(row)

CSV columns: ['county', 'annual_total_mm', 'wettest_month', 'dry_months', 'planting_window']
Number of county records: 5

CSV contents:
{'county': 'Nairobi', 'annual_total_mm': '1045', 'wettest_month': 'April', 'dry_months': '4', 'planting_window': 'April–May'}
{'county': 'Nakuru', 'annual_total_mm': '916', 'wettest_month': 'April', 'dry_months': '3', 'planting_window': 'April–May'}
{'county': 'Kisumu', 'annual_total_mm': '1371', 'wettest_month': 'April', 'dry_months': '0', 'planting_window': 'April–May'}
{'county': 'Mombasa', 'annual_total_mm': '1075', 'wettest_month': 'May', 'dry_months': '2', 'planting_window': 'April–May'}
{'county': 'Machakos', 'annual_total_mm': '811', 'wettest_month': 'November', 'dry_months': '6', 'planting_window': 'November–December'}


## Planting Advisory

The analysis recommends April–May as the best two-month planting window for Nairobi, Nakuru, Kisumu and Mombasa because these months have the highest combined rainfall. Machakos has a different pattern, with November–December identified as its best planting window. Machakos has the highest number of dry months (six), so farmers should plan carefully for water availability. Kisumu recorded the highest annual rainfall at 1,371 mm, while Nakuru recorded the lowest at 916 mm. The NGO should use these findings to guide planting advice while also checking local weather forecasts and soil conditions before planting.

## Data Quality Notes

The original Machakos rainfall data contained two invalid or missing values: April ("N/A") and September (an empty value). These were replaced with 68 mm, the rounded average of the ten valid Machakos readings. This imputation may affect the annual rainfall total and recommended planting window for Machakos. The results should therefore be treated as estimates, and local weather information should also be considered.


In [14]:
with open("requirements.txt", "w") as file:
    file.write("numpy\n")

print("requirements.txt created successfully!")

requirements.txt created successfully!


In [15]:
import os

print("Files created:")
print(os.listdir())

Files created:
['.config', 'requirements.txt', 'rainfall_summary.csv', 'sample_data']
